# Camera-ready SPATIAL experiments — Table 1, San Diego, amplitude sweep, deep baselines
**Upload-and-run on a GPU runtime.** Four blocks:
1. **Pavia Table 1** — the ORIGINAL pipeline (`src/spatial.run_multiseed` +
   `configs/spatial.yaml`: scenario 4, foreign bitumen, n_budget 4000, ρ=0.1,
   1000 epochs, 5 seeds). Prints the per-seed and aggregate summary tables
   (pAUC/AUC/Pd/per-class Pfa) and saves models, figures, raw scores.
2. **Amplitude sweep on Pavia** (θ = 0.03…0.95, same grid as the IID sweep),
   REUSING the Table-1 checkpoints — plus the fixed LRao (robust norm,
   trained once per scene), a global AMF row, and the four deep baselines
   (THANTD, HTD-Net, TSTTD, OS-VAE) trained on the same secondary pixels.
3. **San Diego I–II spatial** — archive-exact boxes + median-norm aircraft
   signature (asserted at build time); same detectors, same sweep.
   AMF-local uses the dimension-aware 21×21 window on the 189-band scenes.
4. **Print-everything cell** + zip download of all results/models.

All datasets ship on this branch (`colab_deep/data/pavia-u.mat`,
`tsp_repro/data/Sandiego*.mat` + region files) — no external downloads.


In [ ]:
!git clone -b tsp-repro --depth 1 https://github.com/michaelpiro/final-paper-experiment.git repo
%cd repo
import sys, os, torch
sys.path.insert(0, '.')
import tsp_repro
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
# datasets are bundled on the branch — verify before spending GPU time
for p in ('colab_deep/data/pavia-u.mat', 'tsp_repro/data/Sandiego.mat',
          'tsp_repro/data/Sandiego2.mat', 'tsp_repro/data/sandiego_regions.json',
          'tsp_repro/data/sandiego2_regions.json', 'tsp_repro/configs/spatial.yaml',
          'tsp_repro/configs/manual_boxes.json'):
    assert os.path.exists(p), f'missing {p}'
print('all bundled datasets present')


In [ ]:
from tsp_repro import spatial_camera_ready as SC
from tsp_repro.iid_camera_ready import apply_robust_lrao, DEEP_BASELINES
apply_robust_lrao()          # fixed LRao for every LRao trained below
PAPER_OVERRIDES = SC.spatial_cfg(DEVICE,
    dataset='colab_deep/data/pavia-u.mat',
    manual_boxes_path='tsp_repro/configs/manual_boxes.json',
    results_dir='results/spatial_pavia')
print({k: PAPER_OVERRIDES[k] for k in ('scenario_index','foreign_class','n_budget',
       'amplitude','dsm_sigma_rho','k','dsm_epochs','nmlp_epochs','cfar_lam',
       'amf_local_window')})


## 1. Pavia Table 1 — original pipeline (5 seeds; hours on T4)
Each seed prints the run_detection summary table; the aggregate mean±std
table and bar figures land in `results/spatial_pavia/multiseed_*/`.


In [ ]:
from src.spatial import run_multiseed
pavia_dir = run_multiseed(PAPER_OVERRIDES, seeds=(42, 43, 44, 45, 46))
print('Pavia Table-1 runs ->', pavia_dir)


## 2. Pavia amplitude sweep (+ fixed LRao, global AMF, deep baselines)
Reuses the Table-1 checkpoints — DART/DARTS are NOT retrained.


In [ ]:
cfg_sweep = SC.spatial_cfg(DEVICE)
res_pavia = SC.run_scene('pavia4', cfg=cfg_sweep, reuse_dir='results/spatial_pavia',
                         deep=DEEP_BASELINES, out_dir='results/spatial_sweep_pavia4')


## 3. San Diego I–II spatial (train + sweep; paper budgets)


In [ ]:
res_sd1 = SC.run_scene('sandiego', cfg=SC.spatial_cfg(DEVICE),
                       deep=DEEP_BASELINES, out_dir='results/spatial_sweep_sandiego')


In [ ]:
res_sd2 = SC.run_scene('sandiego2', cfg=SC.spatial_cfg(DEVICE),
                       deep=DEEP_BASELINES, out_dir='results/spatial_sweep_sandiego2')


## 4. All results in one place


In [ ]:
SC.print_all({'pavia4 (spatial)': res_pavia,
              'sandiego (spatial)': res_sd1,
              'sandiego2 (spatial)': res_sd2})


## Verification — deep baselines in their own regime (REPLACEMENT planting)
Uses ONLY checkpoints already trained in this run (skips seeds with missing
ckpts; nothing retrains). Replacement grid: 0.05...0.95. Healthy ports climb
to ~0.99+ at theta=0.95 (HTD-Net 0.995, THANTD 1.000, OS-VAE 0.996, TSTTD
0.998 in the archived runs); the additive sweep's flatness at strong theta
is the model-mismatch story, not a port failure.


In [ ]:
# === VERIFICATION: deep baselines under the REPLACEMENT model (pretrained ckpts only) ===
import os, numpy as np
from sklearn.metrics import roc_auc_score
from src.data import plant_targets
from src.detectors import amf, dsm_additive
from src.models import score_nmlp_additive
from tsp_repro import spatial_camera_ready as SC
from tsp_repro.iid_camera_ready import _DEEP

REPL_THETAS = [0.05, 0.1, 0.3, 0.5, 0.65, 0.75, 0.85, 0.9, 0.95]

def verify_replacement(scene_name, sweep_dir, ckpt_dir='ckpt_spatial',
                       seeds=(42, 43, 44, 45, 46), deep=DEEP_BASELINES):
    scene = SC.build_scene(scene_name)
    cfgv = SC.spatial_cfg(DEVICE)
    out_dir = os.path.join(sweep_dir, 'replacement_verify')
    os.makedirs(out_dir, exist_ok=True)
    res = {}
    for seed in seeds:
        ours_ck = os.path.join(ckpt_dir, f'ours_{scene_name}_seed{seed}.pt')
        deep_cks = {n: os.path.join(sweep_dir, 'ckpt_deep',
                                    f'{n}__{scene_name}__seed{seed}.pt') for n in deep}
        missing = [n for n, p2 in deep_cks.items() if not os.path.exists(p2)]
        if not os.path.exists(ours_ck): missing.append('ours')
        if missing:
            print(f'seed {seed}: missing ckpts {missing} - SKIPPED (no retraining here)')
            continue
        models = SC.fit_scene_models(scene, cfgv, seed, ckpt_dir, DEVICE)          # resumes
        deep_states = {n: _DEEP[n][0](scene['tr'].astype(np.float64), scene['sig'],
                                      int(seed), p2, DEVICE)                       # resumes
                       for n, p2 in deep_cks.items()}
        if '_te_nbr' not in scene:
            _, scene['_te_nbr'] = SC._windows(scene['te'], scene['te_shape'], int(cfgv['k']), DEVICE)
            _, scene['_tr_nbr'] = SC._windows(scene['tr'], scene['tr_shape'], int(cfgv['k']), DEVICE)
        for th in REPL_THETAS:
            planted, labels, _ = plant_targets(
                scene['te'], scene['sig'], float(th), 0.10, model='replacement',
                seed=int(seed), spatial_shape=scene['te_shape'], edge_guard=3)
            planted = planted.astype(np.float32)
            det_scores = {
                'DART': dsm_additive(planted, scene['tr'], models['dsm'], scene['sig']),
                'DARTS': score_nmlp_additive(models['nmlp'], planted, scene['_te_nbr'],
                                             scene['tr'], scene['_tr_nbr'], scene['sig']),
                'AMF-global': amf(planted, scene['tr'], scene['sig'], eig_floor=0.0),
            }
            for n, st in deep_states.items():
                det_scores[n] = _DEEP[n][1](st, planted.astype(np.float64), scene['sig'], DEVICE)
            np.savez_compressed(os.path.join(
                out_dir, f'scores__{scene_name}__seed{seed}__replacement__{th}.npz'),
                labels=labels.astype(np.int8),
                **{d: np.asarray(s, np.float32) for d, s in det_scores.items()})
            for det, sc_ in det_scores.items():
                au = float(roc_auc_score(labels, sc_))
                res.setdefault(det, {}).setdefault(th, []).append(au)
                print(f'[{scene_name}] seed{seed} {det:10s} REPL th={th}: AUC={au:.4f}', flush=True)
    print(f'\n=== {scene_name} - REPLACEMENT model, AUC mean+-std ===')
    print('| Detector | ' + ' | '.join(f'th={t}' for t in REPL_THETAS) + ' |')
    print('|' + '---|' * (len(REPL_THETAS) + 1))
    for det, r in res.items():
        print(f'| {det} | ' + ' | '.join(
            f'{np.mean(r[t]):.3f}+-{np.std(r[t]):.3f}' if t in r else '--'
            for t in REPL_THETAS) + ' |')
    return res

ver_pavia = verify_replacement('pavia4', 'results/spatial_sweep_pavia4')
# after the SD sweeps finish:
# ver_sd1 = verify_replacement('sandiego',  'results/spatial_sweep_sandiego')
# ver_sd2 = verify_replacement('sandiego2', 'results/spatial_sweep_sandiego2')


In [ ]:
import glob
from IPython.display import Image, display
for d in ('results/spatial_sweep_pavia4', 'results/spatial_sweep_sandiego',
          'results/spatial_sweep_sandiego2'):
    p = glob.glob(f'{d}/figures/auc_vs_theta.png')
    if p: print(p[0]); display(Image(p[0], width=640))


## Zip + download


In [ ]:
from tsp_repro.iid_camera_ready import zip_and_download
zip_and_download(['results/spatial_pavia', 'results/spatial_sweep_pavia4',
                  'results/spatial_sweep_sandiego', 'results/spatial_sweep_sandiego2',
                  'ckpt_spatial'], zip_name='spatial_camera_ready.zip')
